# CNBE-MoE SCNet Jupyter 工作流

适用环境：SCNet Notebook + JupyterLab + Ubuntu 22.04。

本 Notebook 完成环境检查、挂载检查、配置加载、冒烟训练与结果查看；正式多卡训练建议在终端执行 `bash /app/startup.sh`，不要在 Notebook 里阻塞运行长任务。

## 0. 环境确认

A800 场景应看到 `cuda_available=True` 和实际卡数；DCU 场景的 DAS 栈同样兼容 `torch.cuda` API。

In [ ]:
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

import torch

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print("device", i, torch.cuda.get_device_name(i), props.total_memory // (1024**3), "GB")
print("HIP_VISIBLE_DEVICES:", os.environ.get("HIP_VISIBLE_DEVICES"))
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("NPROC_PER_NODE:", os.environ.get("NPROC_PER_NODE"))
print("MASTER_ADDR:", os.environ.get("MASTER_ADDR"))

## 1. 挂载与路径检查

上传包建议挂载：`code -> /app`、`data -> /data/cnbe`、`assets -> /app/assets`、`output -> /output`。

未挂载时 Notebook 会尝试回退到包内相对路径，便于本地调试。

In [ ]:
ROOT = Path(os.environ.get("CNBE_MOE_ROOT", "/app"))
DATA_DIR = Path(os.environ.get("CNBE_DATA_DIR", "/data/cnbe"))
OUTPUT_DIR = Path(os.environ.get("CNBE_OUTPUT_DIR", "/output"))

if not ROOT.exists():
    ROOT = Path("..").resolve()
if not DATA_DIR.exists():
    for cand in (ROOT / "data", Path("/data/cnbe")):
        if cand.exists():
            DATA_DIR = cand
            break
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT", ROOT)
print("DATA_DIR", DATA_DIR)
print("OUTPUT_DIR", OUTPUT_DIR)

required = [
    ROOT / "scripts" / "train_scnet.py",
    ROOT / "scripts" / "train_distributed.py",
    ROOT / "config" / "scnet_moe_config_c.yaml",
    DATA_DIR / "zzjh_294.cnbe",
]
for p in required:
    print("OK " if p.exists() else "MISSING ", p)

## 2. 配置加载

默认使用配置 C：d_model=1024、16 层、256 专家、Top-2、10 epoch。

In [ ]:
import yaml

cfg = yaml.safe_load((ROOT / "config" / "scnet_moe_config_c.yaml").read_text(encoding="utf-8"))
print("experiment:", cfg["experiment_name"])
print("model:", json.dumps(cfg["model"], ensure_ascii=False, indent=2))
print("training:", json.dumps(cfg["training"], ensure_ascii=False, indent=2))
print("data_files:", len(cfg["data"]["cnbe_paths"]))

## 3. 冒烟训练

只使用 `zzjh_294.cnbe`，跑 5 步小模型，验证 DCU/CUDA、数据加载、路由与输出链路。

In [ ]:
smoke_cmd = [
    sys.executable,
    str(ROOT / "scripts" / "train_scnet.py"),
    "--smoke",
    "--cnbe-paths",
    str(DATA_DIR / "zzjh_294.cnbe"),
    "--output",
    str(OUTPUT_DIR / "smoke_metrics.json"),
]
print("RUN:", " ".join(smoke_cmd))
proc = subprocess.run(smoke_cmd, capture_output=True, text=True, timeout=1800)
print(proc.stdout[-6000:])
if proc.stderr:
    print("STDERR_TAIL:", proc.stderr[-2000:])
proc.check_returncode()
print("SMOKE_OK")

## 4. 冒烟结果

重点看：loss 是否下降、专家负载 Gini、吞吐与参数量。

In [ ]:
metrics_path = OUTPUT_DIR / "smoke_metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    print("smoke_metrics.json 尚未生成")

## 5. 正式训练

冒烟通过后，在 Jupyter 的 Terminal 或 SSH 终端执行：

```bash
bash /app/startup.sh
```

`startup.sh` 会自动读取 SCNet 注入的 `MASTER_ADDR/MASTER_PORT/WORLD_SIZE/RANK`，没有时回退单机 standalone。

In [ ]:
nproc = int(os.environ.get("NPROC_PER_NODE", str(max(1, torch.cuda.device_count()))))
torchrun_cmd = (
    f"torchrun --nproc_per_node={nproc} --standalone "
    f"{ROOT / 'scripts' / 'train_distributed.py'} "
    f"--config {ROOT / 'config' / 'scnet_moe_config_c.yaml'} "
    f"--output {OUTPUT_DIR / 'train_metrics.json'} "
    f"--checkpoint-dir {OUTPUT_DIR / 'checkpoints'}"
)
print("正式训练命令（在终端运行）：")
print(torchrun_cmd)

## 6. 结果检查

指标写入 `/output/train_metrics.json`，checkpoint 写入 `/output/checkpoints/`。

In [ ]:
ckpt_dir = OUTPUT_DIR / "checkpoints"
if ckpt_dir.exists():
    files = sorted(ckpt_dir.glob("*.pt"))
    print("checkpoints:", len(files))
    for p in files:
        print(p.name, p.stat().st_size)
else:
    print("no checkpoints yet")